# Solutions · Chapter 02-06 · Looking at two things at once

Worked answers with reasoning. E7 contains the result most people get backwards - the pooled
correlation is *weaker* than either subgroup's - and E14 builds a relationship that reverses sign,
which is the strongest form of the paradox.

Self-contained: run from the top with a fresh kernel.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

trial = pd.DataFrame({
    "station": ["north", "north", "south", "south"],
    "bike": ["electric", "classic", "electric", "classic"],
    "rented": [81, 234, 192, 55], "offered": [87, 270, 263, 80],
})
trial["rate"] = trial["rented"] / trial["offered"]

rng = np.random.default_rng(4)
n = 400
temp = rng.normal(18, 6, n)
weekend = rng.random(n) < 2 / 7
rentals = 40 + 3.0 * temp + 45 * weekend + rng.normal(0, 8, n)
park = pd.DataFrame({"temp_c": temp, "weekend": weekend.astype(int), "rentals": rentals,
                     "staff_on_duty": 6 + 0.05 * rentals + rng.normal(0, 1.2, n)})
print("trial and park ready")

## E1 · Simpson's paradox in one sentence

> A comparison that favours one group in **every** subgroup can reverse when the subgroups are
> pooled.

**The condition it requires: unequal group sizes across a third variable that affects the outcome.**

Both halves are needed. If the third variable does not affect the outcome, pooling changes nothing.
If the groups are equally distributed across it, the pooled comparison is an unweighted average of
comparisons that all point the same way, and cannot reverse. **Balance makes the paradox
arithmetically impossible** - which E4 demonstrates.

## E2 · A perfect relationship with a correlation of 0.82

Because correlation measures closeness to a **straight line**, not the strength of a relationship.

Anscombe's set II lies exactly on a smooth curve - knowing x tells you y with no error at all, which
is as strong as a relationship gets. But the curve bends, so a straight line fits it imperfectly, and
correlation reports that imperfection as 0.82.

**The extreme case makes it obvious:** a perfect, noiseless U-shape, symmetric about its minimum, has
a correlation of **exactly zero**. As x rises, y falls and then rises by the same amount, so the
linear association cancels. A correlation of 0 means "no *linear* association", never "no
relationship".

**The practical consequence:** screening features by correlation with the target - a very common
first step - silently discards every non-monotonic relationship. Tree-based models would have found
them.

## E3 · Why `staff_on_duty` is a bad feature

**Three reasons, and each one alone is disqualifying.**

1. **The causation runs backwards.** Staff are rostered *because* demand is expected. Rentals do not
   go up because someone was scheduled; the schedule went up because rentals were forecast. The
   correlation is real and the arrow points the wrong way.
2. **It will not exist at prediction time.** You decide tomorrow's roster; you do not receive it. A
   model that needs the roster to predict demand cannot run before the roster is set - and the roster
   is the thing the prediction is supposed to inform. This is 00-01's question, and it is fatal on
   its own.
3. **It is a useless lever.** Even if the model used it happily, acting on it - rostering more staff
   to raise demand - does nothing. 00-04's "excellent predictor, useless lever".

**What makes this dangerous rather than merely wrong:** the feature *works*. It correlates at 0.77,
it improves held-out accuracy, and cross-validation applauds. Nothing in the modelling process
objects. Only asking where the number comes from does - which is why 02-02 comes before any model.

**And the legitimate version:** *yesterday's* staffing, or the roster *published a week ahead*, are
both known at prediction time and are fine. The problem is the timing, not the column.

## E4 · What balance would have done

In [ ]:
rates = trial.pivot_table(index="station", columns="bike", values="rate")
observed = trial.groupby("bike").apply(
    lambda g: g["rented"].sum() / g["offered"].sum(), include_groups=False)
balanced = rates.mean()          # equal weight to each station = 175 offered at each

print("rate by station:")
print(rates.round(3).to_string())
print(f"\nas observed  : classic {observed['classic']:.3f}, electric {observed['electric']:.3f} "
      f"-> {'classic' if observed['classic'] > observed['electric'] else 'electric'} wins")
print(f"if balanced  : classic {balanced['classic']:.3f}, electric {balanced['electric']:.3f} "
      f"-> {'classic' if balanced['classic'] > balanced['electric'] else 'electric'} wins")

**As observed:** classic 0.826, electric 0.780 - classic wins.
**Rebalanced to 175 of each type at each station:** classic 0.777, electric 0.831 - **electric wins.**

The per-station rates were not touched. The only thing that changed is the *weights*, and the answer
flipped.

**What that tells you:** the original overall comparison was not measuring the bikes at all. It was
measuring where the bikes happened to be placed. Once each type is weighted equally across stations,
the overall figure agrees with both subgroups - as it must, because a paradox needs unequal weights.

**And the design lesson:** this is exactly why a trial should allocate treatments evenly across the
conditions that matter. Balance is cheaper than any analysis, cannot be got wrong, and removes the
ambiguity that E10 has to argue about.

## E5 · Two surgeries

In [ ]:
surgery = pd.DataFrame({
    "surgery": ["A", "A", "B", "B"],
    "severity": ["severe", "mild", "severe", "mild"],
    "cases": [200, 100, 50, 250],
    "success_rate": [0.70, 0.94, 0.60, 0.88],
})
surgery["successes"] = surgery["cases"] * surgery["success_rate"]

overall = surgery.groupby("surgery").apply(
    lambda g: g["successes"].sum() / g["cases"].sum(), include_groups=False)
print(surgery.to_string(index=False))
print("\noverall success rate:")
print(overall.round(3).to_string())
print("\nby severity:")
print(surgery.pivot_table(index="severity", columns="surgery", values="success_rate").to_string())

- **Surgery A overall:** `(140 + 94) / 300 = 0.780`
- **Surgery B overall:** `(30 + 220) / 300 = 0.833`
- **Severe cases:** A 70%, B 60%. **Mild cases:** A 94%, B 88%.

**A is better for severe patients and better for mild patients, and looks worse overall.**

**Which would I want? A - and it is not close.** Whichever category I am in, A has the higher success
rate. There is no third category to be in.

**Why B looks better:** two thirds of B's cases were mild, and mild cases succeed more whatever you
do. B's overall number is mostly a measurement of *its patient mix*.

**Why severity is a confounder rather than a mediator here**, which is what settles it: severity
exists *before* the surgery and influenced which surgery was chosen - surgeons give the harder
operation to the sicker patients. It is a cause of the assignment, not a consequence of it. So the
within-severity comparison is the honest one.

**This is not a hypothetical.** It is the structure of the classic kidney-stone treatment study, and
the same structure appears whenever outcomes are compared between hospitals, schools or surgeons
without adjusting for who they take on. **A published success rate with no case mix attached is
close to uninterpretable**, and it creates a real incentive to avoid difficult cases.

## E6 · An automated check

In [ ]:
def simpsons_check(df, group, category, numerator, denominator):
    within = df.pivot_table(index=group, columns=category, values=numerator, aggfunc="sum") / \
             df.pivot_table(index=group, columns=category, values=denominator, aggfunc="sum")
    totals = df.groupby(category)[[numerator, denominator]].sum()
    pooled = totals[numerator] / totals[denominator]

    print("within each group:"); print(within.round(3).to_string())
    print("\npooled:"); print(pooled.round(3).to_string())

    winners = set(within.idxmax(axis=1))
    if len(winners) == 1 and winners != {pooled.idxmax()}:
        print(f"\n  ** WARNING: {winners.pop()!r} wins in every group, "
              f"but {pooled.idxmax()!r} wins overall - Simpson's paradox. **")
        share = df.pivot_table(index=category, columns=group, values=denominator, aggfunc="sum")
        print("\n  allocation (this is the cause):")
        print((share.div(share.sum(axis=1), axis=0) * 100).round(0).to_string())
    else:
        print("\n  no reversal: the pooled winner agrees with the groups")


simpsons_check(trial, "station", "bike", "rented", "offered")

The check is deliberately shaped to do two things rather than one.

**It detects the reversal**, by comparing the within-group winner with the pooled winner. That part
is mechanical.

**Then it prints the allocation**, because the reversal is a symptom and the allocation is the cause.
A warning that says only "paradox detected" leaves the reader with a puzzle; showing that 75% of
electric bikes sat at south answers it in the same breath.

**Two honest limits:**

- **It only fires when *every* group agrees.** With three groups where two favour A and one favours
  B, the pooled result can still mislead and this check stays quiet. A fuller version compares each
  group's winner against the pooled one and reports the disagreement rate.
- **It cannot tell you which level is right.** It flags that the question exists. Answering it needs
  to know whether the group variable is a confounder or a mediator, which is not in the data.

**Worth running automatically** on every rate comparison a dashboard makes - the paradox is common,
and nobody looks for it unless something prompts them.

## E7 · Correlation overall, and within each kind of day

In [ ]:
overall_r = park["temp_c"].corr(park["rentals"])
print(f"overall            r = {overall_r:.3f}   (n = {len(park)})")
for value, label in [(0, "weekdays"), (1, "weekends")]:
    sub = park[park["weekend"] == value]
    print(f"{label:<18} r = {sub['temp_c'].corr(sub['rentals']):.3f}   (n = {len(sub)})")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for value, colour, label in [(0, "#0072B2", "weekday"), (1, "#D55E00", "weekend")]:
    m = park["weekend"] == value
    ax.scatter(park.loc[m, "temp_c"], park.loc[m, "rentals"], s=12, alpha=0.6,
               color=colour, label=label)
ax.set_xlabel("Temperature (°C)"); ax.set_ylabel("Rentals per day (count)")
ax.set_title("Two parallel bands: pooling them weakens the correlation")
ax.legend()
plt.show()

| | correlation | n |
|---|---|---|
| Overall | **0.676** | 400 |
| Weekdays | **0.923** | 288 |
| Weekends | **0.893** | 112 |

**Both subgroups correlate more strongly than the pool** - which is the result people find
surprising, because the intuition is that combining data should reveal a relationship rather than
obscure one.

**Why.** The picture shows two parallel bands. Within a band, temperature explains almost everything.
Pool them and the vertical gap between the bands becomes extra spread in `rentals` that temperature
cannot account for - so the same relationship now explains a smaller share of the variation.

**Which would I report? Neither alone.** The honest statement is:

> "Temperature explains most of the day-to-day variation on days of the same kind (r ≈ 0.9). Weekends
> add roughly 45 rentals regardless of temperature, and pooling the two kinds of day masks the
> relationship (r = 0.68)."

**The modelling consequence is concrete.** The two bands are parallel - the *slope* is the same, only
the *level* differs - so `weekend` belongs in the model as an additive feature, not as an
interaction with temperature. Had the bands had different slopes, an interaction term would be
needed. **You can read which one from the picture in two seconds and not at all from the correlation
table**, which is the argument for plotting.

## E8 · "The two strongest pairs are the important relationships"

Three reasons for care, each pointing at an Anscombe set:

1. **A strong correlation can rest on a single point (set IV).** Correlation 0.82, and delete one
   observation and the relationship vanishes entirely. A matrix cannot show how many points support a
   number.
2. **A weak correlation can hide a perfect relationship (set II).** Screening by correlation
   discards every curved and non-monotonic relationship. If the ranking is used to *select* features,
   the strongest real effects can be the ones dropped.
3. **The ranking is distorted by outliers in both directions (set III).** One extreme point can
   inflate a correlation toward a relationship that is not there, or deflate a real one.

**A fourth, from this chapter rather than Anscombe:** the pooled correlation of 0.676 in E7 ranks
*below* two subgroup correlations of 0.9. A matrix computed on mixed populations under-ranks
relationships that are strong within groups - so the "top two" may simply be the two whose
populations happen to be homogeneous.

**What I would say instead:** "these two are worth looking at first - let us plot them, and plot the
five below them too, because the ranking cannot see shape."

## E9 · Retention improved from 61% to 68%, rolled out to enterprise first

**The analysis:** break retention down by customer segment and by period, in one table - old process
versus new, enterprise versus small business.

**What I expect to find:** enterprise customers retain better than small ones regardless of process,
and the new process cohort is disproportionately enterprise. If the within-segment improvement is
much smaller than 7 points - or absent - the headline is a change in customer mix, not a change in
onboarding. This is exactly the trial's structure with "station" replaced by "segment".

**Two further checks before I would believe anything:**

- **A time check.** Was anything else different about the period the new process ran in - a pricing
  change, a seasonal effect, a marketing push? A before/after comparison across a rollout date is a
  changepoint, and 02-02 says a changepoint is a deployment until proven otherwise.
- **Composition inside each segment.** "Enterprise" may itself be a mix; if the new process reached
  the largest enterprise accounts first, the same paradox can hide one level down.

**What would convince me it is real:** the improvement persists **within every segment**, with
enough customers in each for the difference to exceed the noise (03-03), and it survives the period
when both processes ran side by side. Best of all would be the rollout having been **randomised**
within segment - and if the team is still rolling out, that is worth asking for now, because it
turns an argument into a measurement.

**What would not convince me:** a larger overall sample. More customers under the same unbalanced
rollout gives the same biased comparison with a narrower interval - 02-03's lesson.

## E10 · "B is better overall, A is better in every country"

> A wins in every country, so whichever country a user is in, A is the better experience - and the
> overall figure reverses only because the countries had different numbers of users and different
> baseline rates, so B's total is weighted toward whichever countries convert well anyway. The first
> thing I would check is why the allocation was uneven: if country influenced which variant users
> saw, then country is a confounder and the per-country result is the honest one, so we ship A.
> Before shipping I would confirm the pattern is not one large country dominating, and that each
> country has enough users for the difference to be more than noise. And I would fix the experiment
> design, because a properly randomised test allocates variants evenly within each country and this
> ambiguity cannot arise - the fact that it did tells me the assignment was not random, which is worth
> knowing for every other result from the same test.

**What is being assessed:** whether you recognise the structure, and whether you go to the *cause* -
uneven allocation - rather than just picking a level. The best answers also notice that the paradox
is evidence the randomisation was broken, which casts doubt on the whole experiment rather than only
this metric.

## E11 · Sixty columns, thirty minutes

**In order:**

1. **Two minutes - shape and dtypes.** `df.info()`, row count, and what one row represents (02-01).
   Everything else is meaningless until that sentence exists.
2. **Five minutes - a quality profile.** Missing rate, distinct count, min/max, most common value per
   column (02-04's `quality_report`). This finds the sentinels, the impossible values and the
   constant columns immediately.
3. **Five minutes - the target.** Its distribution, its skew, its extremes, and whether it is one
   population or two (02-05). If there is no target yet, that is the finding and I stop to ask.
4. **Ten minutes - ten scatters, not sixteen hundred.** The ten columns a domain expert says matter,
   plotted against the target, coloured by the most obvious grouping variable.
5. **Five minutes - the time dimension.** Every important column plotted over time, looking for steps,
   gaps and level shifts (02-02).
6. **Three minutes - write it down.** Three sentences on what one row is, three surprises, three
   questions for whoever owns the data.

**What I would deliberately not do:**

- **A 60x60 correlation heatmap.** 1,770 numbers, unreadable, and every one of them blind to
  curvature and leverage.
- **A full scatter matrix.** 3,600 panels, each too small to see.
- **Impute or clean anything**, before knowing why values are missing.
- **Fit a model.** Thirty minutes of exploration exists to produce good questions, and a model
  answers a question you have not asked yet.

**The output is not a notebook of charts. It is a page of sentences**, and the most valuable line on
it usually begins "I do not understand why...".

## E12 · Lurking variables

| | The lurking variable to look for first |
|---|---|
| (a) Department appears to admit men at a higher rate | **Which programme people applied to.** Women may apply in greater numbers to more competitive programmes. This is the Berkeley admissions case, the most famous real Simpson's paradox |
| (b) Hospital with a higher death rate | **Case mix and severity.** A specialist centre takes the hardest cases; a hospital that refuses them will always look better |
| (c) Channel with the lowest cost per acquisition | **Who was already going to convert.** Branded search and retargeting capture existing intent, so they take credit for customers who would have arrived anyway |
| (d) School whose results fell after a new curriculum | **The intake.** A changed catchment, a new selection rule, or a different cohort size. Also the exam itself, which may have been re-standardised |
| (e) Route with a high damage rate | **What is being carried, and how far.** A route handling fragile goods or long distances damages more regardless of how it is driven |

**Every one of these has the same shape:** the groups being compared were not composed the same way,
and the composition is related to the outcome. **The question that finds all five is "who ended up in
each group, and why?"** - which is the provenance question again.

**And note the incentive in (b) and (d):** where a published comparison ignores case mix, the way to
improve your ranking is to take on easier cases. A metric that can be gamed by selection will be.

## E13 · Explaining it to Maria

> Most of the electric bikes were sitting at south, and south is your quiet station - both kinds of
> bike get rented less there. Most of the classic bikes were at north, where everything gets rented.
> So the two totals are really comparing your two stations, not your two kinds of bike. Put the same
> number of each at each station and the electric ones come out ahead.

(65 words.)

**Why it lands:** it never mentions arithmetic. Maria knows her stations, so once she pictures where
the bikes were sitting, the conclusion follows from something she already knows. The last sentence
also tells her what to do next - run the trial evenly - which turns an explanation into a decision.

## E14 · A relationship that reverses sign

In [ ]:
gen = np.random.default_rng(8)
records = []
for station, size, base_rate in [("A", 12, 0.92), ("B", 40, 0.74), ("C", 90, 0.58)]:
    for _ in range(120):
        wobble = gen.normal(0, 1.5)
        records.append({
            "station": station,
            "staff": size * 0.09 + 0.5 * wobble + 1,            # big stations have more staff
            "rate_per_bike": base_rate + 0.030 * wobble + gen.normal(0, 0.012),
        })
depot = pd.DataFrame(records)

overall_slope = np.polyfit(depot["staff"], depot["rate_per_bike"], 1)[0]
print(f"pooled : slope {overall_slope:+.4f} per extra staff member "
      f"(r = {depot['staff'].corr(depot['rate_per_bike']):+.2f})")
for station, group in depot.groupby("station"):
    print(f"   {station}  : slope {np.polyfit(group['staff'], group['rate_per_bike'], 1)[0]:+.4f} "
          f"(r = {group['staff'].corr(group['rate_per_bike']):+.2f})")

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5))
line_x = np.array([depot["staff"].min(), depot["staff"].max()])
for station, colour in zip("ABC", ["#0072B2", "#D55E00", "#009E73"]):
    g = depot[depot["station"] == station]
    ax.scatter(g["staff"], g["rate_per_bike"], s=10, alpha=0.6, color=colour, label=f"station {station}")
    slope, intercept = np.polyfit(g["staff"], g["rate_per_bike"], 1)
    gx = np.array([g["staff"].min(), g["staff"].max()])
    ax.plot(gx, intercept + slope * gx, color=colour, linewidth=2)

slope, intercept = np.polyfit(depot["staff"], depot["rate_per_bike"], 1)
ax.plot(line_x, intercept + slope * line_x, color="black", linewidth=2.5, linestyle="--",
        label="pooled fit (negative)")
ax.set_xlabel("Staff on duty"); ax.set_ylabel("Rentals per bike available")
ax.set_title("Positive within every station, negative when pooled")
ax.legend(fontsize=8)
plt.show()

**Pooled slope: -0.041 per extra staff member (r = -0.82). Within each station: +0.057 to +0.061
(r = +0.97).**

Not a weakening - a **sign reversal**. Pooled, the data says more staff means fewer rentals per bike.
Within every station it says the opposite, strongly.

**The mechanism, visible in the picture as three clouds on a downward diagonal:** big stations have
more staff *and* lower rentals per bike available, because a 90-bike rack cannot rent out the same
proportion as a 12-bike one. Station size drives both variables in opposite directions, and the
pooled line simply traces the three station clouds from top-left to bottom-right. It is not
measuring staffing at all - it is measuring station size.

**Which fit answers "should we roster more staff?" The within-station one.**

That is the question of what happens if you *intervene* - add a person to a station whose size does
not change. Only the within-station relationship holds station size fixed, so only it can answer it.
The pooled line answers a different question - "what are stations with more staff like?" - and
stations with more staff are big ones.

**The general form, and it is the whole of module 02 in one sentence:**

> **A relationship between two columns is not a fact about those two columns.** It is a fact about
> those two columns *and* everything you held constant, which includes everything you did not think
> to hold constant.

The name for the pooled line's error is **aggregation bias**, and it is Simpson's paradox on
continuous variables. The defence is the same: break it down before you believe it, and know why the
groups differ.

---

## Where to go next

Back to the chapter for the mastery check and flashcards, then **02-07 · Honest visualisation,
correlation versus causation, and the limits of EDA**.